# Clase 2 · Tipos de datos y operadores en Python


### Ciclo de formación para semilleristas · SIPIAD
**Enfoque:** Inteligencia Artificial y Ciencia de Datos (Machine Learning y Deep Learning)
---



## Objetivos de aprendizaje
Al finalizar la clase, el estudiante será capaz de:

1. Identificar y manipular los **tipos de datos fundamentales** de Python y entender cómo se relacionan con los tipos de **NumPy/pandas** que usan los modelos de ML/DL.


2. Aplicar los **operadores aritméticos, de comparación, de asignación, lógicos, de pertenencia, de identidad y bit a bit** en problemas típicos de IA.


3. Reconocer **errores sutiles** (precisión de flotantes, `NaN`, desbordamiento de `uint8`, alias de listas, `and` vs `&`) que causan bugs reales en pipelines de datos.


4. Integrar todo en un **mini-proyecto**: limpiar datos, normalizarlos y entrenar una regresión lineal con descenso del gradiente usando únicamente operadores.



## Agenda sugerida (≈ 3 horas)

| Bloque | Contenido | Tiempo |
|---|---|---|
| 0 | Entorno y calentamiento | 10 min |
| 1 | Tipos de datos escalares y su rol en ML | 30 min |
| 2 | Colecciones: `list`, `tuple`, `set`, `dict`, `range` | 25 min |
| 3 | Tipos de datos en NumPy y pandas (`dtype`) | 15 min |
| 4 | Operadores aritméticos | 25 min |
| 5 | Operadores de comparación | 15 min |
| 6 | Operadores de asignación | 15 min |
| 7 | Operadores lógicos | 20 min |
| 8 | Pertenencia e identidad | 15 min |
| 9 | Operadores bit a bit y precedencia | 10 min |
| 10 | 🧪 Mini-proyecto integrador | 30 min |
| 11 | Ejercicios, autoevaluación y cierre | 20 min |



> 💡 **Cómo usar este cuaderno:** ejecuta cada celda con `Shift + Enter`, **predice el resultado antes de correrla** y modifica los valores para experimentar.

## Preparación del entorno
Usaremos solo bibliotecas estándar del ecosistema de ciencia de datos: **NumPy**, **pandas** y **Matplotlib**.
Si trabajas en Google Colab ya vienen instaladas; en local ejecuta `pip install numpy pandas matplotlib`.

In [ ]:
import sys, math, time, copy, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Semillas: la reproducibilidad es una obligación en ciencia de datos
SEED = 42
random.seed(SEED)
rng = np.random.default_rng(SEED) # esta línea sirve para generar números 
#aleatorios reproducibles en NumPy

print("Python :", sys.version.split()[0])
print("NumPy  :", np.__version__)
print("pandas :", pd.__version__)

---
## 1️⃣ Tipos de datos escalares

En Python **todo es un objeto** y cada objeto tiene un **tipo**. Python es de *tipado dinámico*: no declaras el tipo, el intérprete lo deduce del valor.

| Tipo | Ejemplo | Uso típico en IA / Ciencia de datos |
|---|---|---|
| `int` | `32` | tamaño de batch, número de épocas, índices, etiquetas de clase |
| `float` | `0.001` | tasa de aprendizaje, pesos, pérdidas, probabilidades |
| `complex` | `3+4j` | procesamiento de señales (FFT), análisis espectral |
| `bool` | `True` | máscaras, banderas (`entrenable`, `es_valido`) |
| `str` | `"ResNet-50"` | nombres de modelos, etiquetas, texto para NLP |
| `NoneType` | `None` | "sin valor": parámetros opcionales, valores ausentes |

Usamos `type()` para consultar el tipo y `isinstance()` para verificarlo.

In [ ]:
batch_size      = 32            # int
tasa_aprendizaje = 0.001        # float
senal           = 3 + 4j        # complex
es_entrenable   = True          # bool
nombre_modelo   = "ResNet-50"   # str
checkpoint      = None          # NoneType

for valor in [batch_size, tasa_aprendizaje, senal, es_entrenable, nombre_modelo, checkpoint]:
    print(f"{valor!r:>12}  →  {type(valor).__name__}")

print()
print("¿batch_size es int?     ", isinstance(batch_size, int))
print("¿tasa es int o float?   ", isinstance(tasa_aprendizaje, (int, float)))

### 1.1 Enteros (`int`): precisión arbitraria
A diferencia de C o Java, los `int` de Python **no se desbordan**: crecen tanto como la memoria permita. Útil para combinatoria, hashing y conteos enormes (p. ej. número de parámetros de un LLM).

In [ ]:
print(2 ** 100)                                  # entero gigante, sin overflow
parametros_gpt = 175_000_000_000                 # _ separa miles para legibilidad
print(f"{parametros_gpt:,} parámetros")
print(type(parametros_gpt))

### 1.2 Flotantes (`float`): ¡la fuente #1 de bugs numéricos!
Los `float` de Python son de **64 bits (IEEE 754, doble precisión)**. No pueden representar exactamente muchos decimales (como 0.1), por eso aparecen errores de redondeo. En ML esto importa: pérdidas, gradientes y probabilidades son flotantes.

In [ ]:
print(0.1 + 0.2)                 # 0.30000000000000004
print(0.1 + 0.2 == 0.3)          # False  ← ¡cuidado!
print(math.isclose(0.1 + 0.2, 0.3))   # ✅ forma correcta de comparar flotantes

print("\nLímites de float64:")
print("  máximo   :", sys.float_info.max)
print("  épsilon  :", sys.float_info.epsilon)   # menor diferencia relativa representable

print("\nValores especiales:")
inf, nan = float("inf"), float("nan")
print("  inf  :", inf, "| 1e308 * 10 =", 1e308 * 10)
print("  nan  :", nan, "| nan == nan →", nan == nan)   # ¡NaN no es igual a sí mismo!
print("  ¿es NaN? →", math.isnan(nan))

> 🧠 **Conexión con Deep Learning:** las redes neuronales suelen entrenarse en `float32` (o `float16`/`bfloat16` en GPU) para ahorrar memoria y acelerar el cómputo. Menos bits = menos precisión **y menor rango**.

In [ ]:
# Precisión: el mismo "0.1" almacenado con distintas precisiones
print(f"float64: {0.1:.25f}")
print(f"float32: {np.float32(0.1):.25f}")
print(f"float16: {np.float16(0.1):.25f}")

# Rango: el máximo representable en cada formato
print("\nValor máximo representable:")
for t in (np.float16, np.float32, np.float64):
    print(f"  {t.__name__:8s} → {np.finfo(t).max:.3e}")

# Desbordamiento en float16 (típico al entrenar con precisión mixta)
with np.errstate(over="ignore"):
    print("\nfloat16(65504) + float16(100) =", np.float16(65504) + np.float16(100))

### 1.3 Booleanos (`bool`): son un subtipo de `int`
`True` vale `1` y `False` vale `0`. Esto permite **contar** y **promediar** condiciones: la base de métricas como la *accuracy*.

In [ ]:
print(isinstance(True, int))      # True: bool hereda de int
print(True + True)                # 2
print(sum([True, False, True, True]))   # 3 → cuenta de aciertos

# Accuracy = promedio de aciertos (True=1, False=0)
y_real = np.array([1, 0, 1, 1, 0, 1, 0, 0])
y_pred = np.array([1, 0, 0, 1, 0, 1, 1, 0])
aciertos = (y_real == y_pred)
print("\naciertos:", aciertos)
print("accuracy:", aciertos.mean())      # 6/8 = 0.75

### 1.4 Cadenas (`str`): inmutables y esenciales en NLP
Las cadenas son **inmutables**: sus métodos devuelven una *nueva* cadena. Son la materia prima del procesamiento de lenguaje natural.

In [ ]:
texto = "El modelo aprende patrones a partir de los datos"

print(texto.lower())
print(texto.split())                   # tokenización ingenua por espacios
print(len(texto.split()), "tokens")
print(texto.replace("modelo", "algoritmo"))
print(texto[3:9], "|", texto[::-1][:10])   # indexación y rebanado (slicing)

# f-strings: la forma moderna de formatear salidas
epoch, loss, acc = 7, 0.31415, 0.9234
print(f"Época {epoch:03d} | loss={loss:.4f} | acc={acc:.2%}")

# Inmutabilidad
try:
    texto[0] = "e"
except TypeError as e:
    print("\nTypeError:", e)

### 1.5 `None`: la ausencia de valor
Es un objeto único (singleton) que significa "nada". Se compara con `is`, no con `==` (lo veremos en el bloque 8).

In [ ]:
def entrenar(modelo, optimizador=None, epochs=10):
    if optimizador is None:
        optimizador = "sgd"       # valor por defecto
    return f"Entrenando {modelo} con {optimizador} por {epochs} épocas"

print(entrenar("MLP"))
print(entrenar("CNN", optimizador="adam", epochs=50))

### 1.6 Conversión de tipos (*casting*)
Pasar datos "crudos" (casi siempre `str` al leer CSV/JSON) al tipo correcto es el primer paso de cualquier pipeline.

In [ ]:
print(int("42"), float("3.14"), str(2025), list("IA"))
print(int(3.99), round(3.99), math.floor(-3.5), math.ceil(-3.5))   # int() trunca hacia 0
print(bool(0), bool(0.0), bool(""), bool([]), bool(None))          # "falsy"
print(bool(1), bool("0"), bool([0]), bool(-1))                     # "truthy"

# Un error clásico al leer datos sucios:
try:
    int("3.7")
except ValueError as e:
    print("\nValueError:", e)
print("Solución:", int(float("3.7")))

# Parseo defensivo de una columna con basura
crudo = ["12.5", "8", "N/A", "", "20.1", "?"]
limpio = []
for dato in crudo:
    try:
        limpio.append(float(dato))
    except ValueError:
        limpio.append(float("nan"))     # marcamos como dato faltante
print("\nlimpio:", limpio)

---
## 2️⃣ Colecciones: estructuras de datos integradas

| Tipo | Sintaxis | Ordenado | Mutable | Duplicados | Uso en IA |
|---|---|:-:|:-:|:-:|---|
| `list` | `[1, 2, 3]` | ✅ | ✅ | ✅ | secuencia de muestras, historial de *loss* |
| `tuple` | `(224, 224, 3)` | ✅ | ❌ | ✅ | `shape` de tensores, pares (X, y) |
| `set` | `{"a", "b"}` | ❌ | ✅ | ❌ | clases únicas, vocabulario |
| `dict` | `{"lr": 0.01}` | ✅ (desde 3.7) | ✅ | claves únicas | hiperparámetros, configuración, JSON |
| `range` | `range(10)` | ✅ | ❌ | — | iterar épocas / batches |

In [ ]:
historial_loss = [0.92, 0.61, 0.43, 0.35]            # list
shape_imagen   = (224, 224, 3)                       # tuple
clases         = {"gato", "perro", "gato", "ave"}    # set (descarta duplicados)
hiperparams    = {"lr": 0.001, "batch": 32, "optim": "adam"}   # dict

historial_loss.append(0.31)           # las listas se modifican
print(historial_loss)
print(clases)                          # solo 3 elementos únicos
print(hiperparams["lr"], hiperparams.get("dropout", 0.0))   # get() evita KeyError

try:
    shape_imagen[0] = 128
except TypeError as e:
    print("TypeError:", e)             # las tuplas son inmutables

print(list(range(0, 10, 2)))           # inicio, fin (exclusivo), paso
alto, ancho, canales = shape_imagen    # desempaquetado de tupla
print(f"{alto}x{ancho} con {canales} canales")

### 2.1 Un dataset "a mano": lista de diccionarios → DataFrame
Antes de pandas, así se representaría una tabla en Python puro. Casi todos los datos de la web (JSON/APIs) llegan con esta forma.

In [ ]:
estudiantes = [
    {"id": 1, "horas_estudio": 5.5, "asistencia": 0.92, "aprobo": True},
    {"id": 2, "horas_estudio": 2.0, "asistencia": 0.60, "aprobo": False},
    {"id": 3, "horas_estudio": 8.0, "asistencia": 0.98, "aprobo": True},
    {"id": 4, "horas_estudio": 1.0, "asistencia": 0.45, "aprobo": False},
]

print(estudiantes[0]["horas_estudio"])
print([e["horas_estudio"] for e in estudiantes])     # "columna" como lista

df = pd.DataFrame(estudiantes)
display(df)
print(df.dtypes)

---
## 3️⃣ Tipos de datos en NumPy y pandas (`dtype`)

Los tipos nativos de Python son flexibles pero **lentos y gastan memoria**. Las librerías de ML usan arreglos homogéneos con un `dtype` fijo:

| Python | NumPy | Equivalente en PyTorch / TensorFlow | Uso |
|---|---|---|---|
| `float` | `float64` | `torch.float64` | cálculo científico |
| — | `float32` | `torch.float32` / `tf.float32` | **entrenamiento estándar** |
| — | `float16` / `bfloat16` | `torch.float16` | precisión mixta, inferencia |
| `int` | `int64` | `torch.int64` | etiquetas, índices |
| — | `uint8` | `torch.uint8` | **píxeles de imagen (0–255)** |
| `bool` | `bool_` | `torch.bool` | máscaras |

In [ ]:
a = np.array([1, 2, 3])
b = np.array([1, 2.5, 3])           # un float "contagia" a todo el arreglo (upcasting)
c = np.array([1, 2, 3], dtype=np.float32)
print(a.dtype, b.dtype, c.dtype)

# Ahorro de memoria: 1 millón de números
n = 1_000_000
for t in (np.float64, np.float32, np.float16):
    arr = np.zeros(n, dtype=t)
    print(f"{t.__name__:8s} → {arr.nbytes / 1e6:.1f} MB")

#### ⚠️ Trampa clásica en visión por computador: desbordamiento de `uint8`
Las imágenes se guardan como enteros de 8 bits (0–255). Si sumas brillo sin convertir el tipo, el valor **da la vuelta** (*wrap-around*) sin avisar.

In [ ]:
pixeles = np.array([250, 251, 252], dtype=np.uint8)
print("uint8 + 10      :", pixeles + 10)                       # ¡[4 5 6]! overflow silencioso
print("int16 + 10      :", pixeles.astype(np.int16) + 10)      # ✅ [260 261 262]
print("normalizado /255:", pixeles / 255.0)                    # float64 en [0, 1]

#### Tipos en pandas
`DataFrame.dtypes` te dice qué hay en cada columna. Convertir texto repetitivo a `category` ahorra memoria y es lo que consumen muchos modelos.

In [ ]:
df2 = pd.DataFrame({
    "edad":      [23, 35, 41, 29],
    "ingreso":   [1.8e6, 3.2e6, 2.5e6, 4.1e6],
    "ciudad":    ["Medellín", "Bogotá", "Medellín", "Cali"],
    "activo":    [True, False, True, True],
})
print(df2.dtypes, "\n")

df2["ciudad"] = df2["ciudad"].astype("category")
df2["ingreso"] = df2["ingreso"].astype("float32")
print(df2.dtypes)
print("\nCódigos numéricos de la categoría:", df2["ciudad"].cat.codes.tolist())

---
## 4️⃣ Operadores aritméticos

| Operador | Nombre | Ejemplo | Resultado |
|:-:|---|---|---|
| `+` | Suma | `7 + 3` | `10` |
| `-` | Resta | `7 - 3` | `4` |
| `*` | Multiplicación | `7 * 3` | `21` |
| `/` | División real | `7 / 2` | `3.5` |
| `//` | División entera (piso) | `7 // 2` | `3` |
| `%` | Módulo (resto) | `7 % 2` | `1` |
| `**` | Potencia | `2 ** 10` | `1024` |
| `@` | Multiplicación matricial | `A @ B` | (NumPy) |

In [ ]:
a, b = 7, 2
print("a + b  =", a + b)
print("a - b  =", a - b)
print("a * b  =", a * b)
print("a / b  =", a / b)        # SIEMPRE devuelve float en Python 3
print("a // b =", a // b)
print("a % b  =", a % b)
print("a ** b =", a ** b)
print("divmod(7, 2) =", divmod(a, b))     # (cociente, resto) en un paso

print("\nCon negativos (¡// redondea hacia -∞!):")
print("-7 // 2 =", -7 // 2, "| -7 % 3 =", -7 % 3)
print("\nPotencias raras: 2 ** -1 =", 2 ** -1, "| 9 ** 0.5 =", 9 ** 0.5)

### 4.1 Aplicaciones en ML: **normalización de características**
Los modelos (regresión, redes, k-NN, SVM) funcionan mejor cuando las variables están en escalas comparables.

- **Min-Max:**  $x' = \dfrac{x - x_{min}}{x_{max} - x_{min}}$  → rango [0, 1]
- **Z-score:**  $z = \dfrac{x - \mu}{\sigma}$  → media 0, desviación 1

In [ ]:
edades = [18, 25, 32, 47, 60]

# --- Puro Python ---
x_min, x_max = min(edades), max(edades)
minmax = [(x - x_min) / (x_max - x_min) for x in edades]

mu = sum(edades) / len(edades)
sigma = (sum((x - mu) ** 2 for x in edades) / len(edades)) ** 0.5
zscore = [(x - mu) / sigma for x in edades]

print("min-max:", [round(v, 3) for v in minmax])
print("z-score:", [round(v, 3) for v in zscore])

# --- NumPy: operaciones vectorizadas (sin bucles) ---
x = np.array(edades, dtype=float)
print("\nz-score NumPy:", ((x - x.mean()) / x.std()).round(3))

### 4.2 Aplicaciones en ML: **función de pérdida (MSE)** y **norma L2**
$$MSE = \frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2 \qquad\qquad \|w\|_2 = \sqrt{\sum_i w_i^2}$$

In [ ]:
y_real = np.array([3.0, 5.0, 2.5, 7.0])
y_pred = np.array([2.8, 5.4, 2.0, 8.0])

error = y_real - y_pred
mse   = (error ** 2).mean()
rmse  = mse ** 0.5
mae   = abs(error).mean()
print(f"MSE={mse:.4f} | RMSE={rmse:.4f} | MAE={mae:.4f}")

pesos = np.array([0.5, -1.2, 3.0, 0.0])
norma_l2 = (pesos ** 2).sum() ** 0.5          # regularización L2 / "weight decay"
print(f"‖w‖₂ = {norma_l2:.4f}")

### 4.3 `//` y `%` en el entrenamiento: batches y *logging*
- `//` calcula **cuántos batches completos** caben en el dataset.
- `%` decide **cada cuánto** hacer algo (guardar un checkpoint, imprimir).

In [ ]:
n_muestras, batch = 1000, 32
completos = n_muestras // batch            # 31
sobrante  = n_muestras % batch             # 8
total     = math.ceil(n_muestras / batch)  # 32 (incluyendo el último incompleto)
print(f"{completos} batches completos + {sobrante} muestras sobrantes → {total} iteraciones/época")

for epoch in range(1, 31):
    if epoch % 10 == 0:                    # solo cada 10 épocas
        print(f"  ✔ checkpoint guardado en la época {epoch}")

### 4.4 Los operadores se comportan distinto según el tipo
`+` y `*` **concatenan/repiten** en listas, pero **operan elemento a elemento** en NumPy. Confundirlos es un error muy común.

In [ ]:
l1, l2 = [1, 2, 3], [10, 20, 30]
print("listas  l1 + l2  :", l1 + l2)          # concatena
print("listas  l1 * 2   :", l1 * 2)           # repite
print("listas  [0] * 5  :", [0] * 5)          # idioma para inicializar

n1, n2 = np.array(l1), np.array(l2)
print("\nNumPy   n1 + n2  :", n1 + n2)        # suma elemento a elemento
print("NumPy   n1 * 2   :", n1 * 2)
print("NumPy   n1 * n2  :", n1 * n2)          # Hadamard (elemento a elemento)
print("NumPy   n1 @ n2  :", n1 @ n2)          # producto punto = 1*10+2*20+3*30

### 4.5 *Broadcasting* y el operador `@`: una neurona artificial
Una neurona calcula $\hat{y} = \sigma(\mathbf{x}\cdot\mathbf{w} + b)$ con $\sigma(z)=\dfrac{1}{1+e^{-z}}$.
Con operadores aritméticos y `@` se programa en **una línea**.

In [ ]:
def sigmoide(z):
    return 1 / (1 + np.exp(-z))

# 4 muestras con 3 características cada una
X = np.array([[0.5, 1.0, -0.3],
              [1.5, -0.5, 0.8],
              [0.0, 0.0, 0.0],
              [2.0, 1.0, 1.0]])
w = np.array([0.4, -0.6, 0.9])     # pesos
b = 0.1                             # sesgo (bias): se "propaga" a las 4 filas (broadcasting)

z = X @ w + b
print("z      =", z.round(3))
print("ŷ=σ(z) =", sigmoide(z).round(3))

# Capa densa completa: 4 muestras → 5 neuronas
W = rng.normal(size=(3, 5))
b_vec = np.zeros(5)
salida = np.maximum(0, X @ W + b_vec)     # ReLU = max(0, z)
print("\nForma de la salida de la capa:", salida.shape)

---
## 5️⃣ Operadores de comparación
Devuelven un `bool` (o un arreglo de `bool` en NumPy/pandas).

| Operador | Significado |
|:-:|---|
| `==` | igual a |
| `!=` | distinto de |
| `>` , `<` | mayor / menor que |
| `>=` , `<=` | mayor o igual / menor o igual |

In [ ]:
x = 5
print(x == 5, x != 5, x > 3, x < 3, x >= 5, x <= 4)

# Comparación encadenada (exclusiva de Python): ¡muy legible!
p = 0.73
print("¿p es una probabilidad válida?", 0 <= p <= 1)

# También funcionan con str, listas y tuplas (orden lexicográfico)
print("abc" < "abd", [1, 2, 3] < [1, 2, 4], (2, 0) > (1, 9))

### 5.1 Comparaciones y **máscaras booleanas** (la herramienta #1 para filtrar datos)

In [ ]:
probs = np.array([0.91, 0.12, 0.55, 0.49, 0.77, 0.03])
y_true = np.array([1, 0, 1, 1, 1, 0])

umbral = 0.5
y_hat = (probs >= umbral).astype(int)        # clasificación binaria por umbral
print("predicción:", y_hat)
print("real      :", y_true)
print("aciertos  :", y_hat == y_true)
print(f"accuracy  : {(y_hat == y_true).mean():.2%}")

# Filtrado con máscara
mascara = probs > 0.5
print("\nProbabilidades > 0.5:", probs[mascara])
print("Índices         > 0.5:", np.where(mascara)[0])

### 5.2 Trampas con la comparación
1. **Flotantes:** no uses `==`; usa `math.isclose` / `np.allclose`.
2. **`NaN`:** `nan == nan` es `False`; usa `np.isnan` / `pd.isna`.
3. **Arreglos en un `if`:** `if a == b` lanza `ValueError`; usa `.all()`, `.any()` o `np.array_equal`.

In [ ]:
# 1) Flotantes
a, b = 0.1 * 3, 0.3
print(a == b, "|", math.isclose(a, b), "|", np.allclose([a], [b]))

# 2) NaN
vals = np.array([1.0, np.nan, 3.0])
print("\nvals == nan  :", vals == np.nan)       # [False False False] ← engañoso
print("np.isnan     :", np.isnan(vals))         # ✅

# 3) Arreglos
u, v = np.array([1, 2, 3]), np.array([1, 2, 4])
try:
    if u == v:
        pass
except ValueError as e:
    print("\nValueError:", str(e)[:70], "...")
print("all:", (u == v).all(), "| any:", (u == v).any(), "| array_equal:", np.array_equal(u, v))

---
## 6️⃣ Operadores de asignación

| Operador | Equivale a | | Operador | Equivale a |
|:-:|---|---|:-:|---|
| `=` | asigna | | `//=` | `x = x // y` |
| `+=` | `x = x + y` | | `%=` | `x = x % y` |
| `-=` | `x = x - y` | | `**=` | `x = x ** y` |
| `*=` | `x = x * y` | | `:=` | **walrus**: asigna *dentro* de una expresión |
| `/=` | `x = x / y` | | | |

In [ ]:
# Asignación múltiple y desempaquetado
lr, epochs, optim = 0.01, 100, "adam"
a = b = c = 0                      # los tres valen 0
a, b = b, a                        # intercambio sin variable temporal
primero, *medio, ultimo = [10, 20, 30, 40, 50]
print(primero, medio, ultimo)

# Asignación aumentada
contador = 0
contador += 1; contador *= 10; contador -= 4; contador //= 2; contador **= 2
print("contador =", contador)

# Acumular pérdida promedio durante una época
perdidas_batch = [0.9, 0.8, 0.85, 0.7]
suma = 0.0
for p in perdidas_batch:
    suma += p
print(f"pérdida media de la época = {suma / len(perdidas_batch):.4f}")

### 6.1 El corazón del aprendizaje: **descenso del gradiente**
Regla de actualización: $w \leftarrow w - \eta \cdot \nabla f(w)$. En Python es literalmente un `-=`.

Minimicemos $f(w)=(w-3)^2$, cuya derivada es $f'(w)=2(w-3)$. El mínimo está en $w=3$.

In [ ]:
w = -4.0           # punto inicial arbitrario
eta = 0.1          # tasa de aprendizaje (learning rate)
trayectoria = [w]

for paso in range(1, 26):
    gradiente = 2 * (w - 3)
    w -= eta * gradiente           # ← la línea que "entrena"
    trayectoria.append(w)
    if paso % 5 == 0:
        print(f"paso {paso:2d} | w = {w:.5f} | f(w) = {(w - 3) ** 2:.6f}")

plt.figure(figsize=(6, 3))
plt.plot(trayectoria, "o-")
plt.axhline(3, color="r", ls="--", label="mínimo (w=3)")
plt.xlabel("paso"); plt.ylabel("w"); plt.legend(); plt.title("Convergencia del gradiente descendente")
plt.tight_layout(); plt.show()

### 6.2 El operador *walrus* `:=` (Python ≥ 3.8)
Asigna y usa el valor en la misma expresión. Evita calcular dos veces.

In [ ]:
datos = [3, 8, 1, 9, 4]
if (n := len(datos)) > 3:
    print(f"Hay {n} datos: suficientes para entrenar")

# Útil en bucles: leer hasta que ocurra una condición
valores = iter([0.5, 0.3, 0.1, 0.001, 0.0001])
while (loss := next(valores)) > 0.01:
    print(f"  loss = {loss} → seguir entrenando")
print(f"  loss = {loss} → ¡convergió!")

### 6.3 ⚠️ Asignar **no copia**: alias y mutabilidad
En objetos mutables (`list`, `dict`, `ndarray`), `b = a` crea **otro nombre para el mismo objeto**.
Un error típico: modificar el dataset original sin querer.

In [ ]:
# Listas
pesos = [0.1, 0.2]
referencia = pesos                 # ALIAS (mismo objeto)
referencia.append(0.3)
print("pesos original:", pesos)    # ¡cambió!

copia = pesos.copy()               # copia superficial
copia.append(99)
print("pesos tras copia:", pesos, "| copia:", copia)

# Anidados: se necesita deepcopy
anidada = [[1, 2], [3, 4]]
superficial = anidada.copy()
profunda = copy.deepcopy(anidada)
anidada[0][0] = 999
print("\nsuperficial:", superficial, "(¡también cambió!)")
print("profunda   :", profunda)

# NumPy
x = np.arange(5)
y = x                              # alias
y[0] = 99
print("\nx =", x)                  # x cambió
z = x.copy(); z[1] = -1
print("x tras copy() =", x)

# In-place con dtype distinto
v = np.array([1, 2, 3])
try:
    v += 0.5                       # int64 += float → no permitido
except TypeError as e:
    print("\n", type(e).__name__, "→ usa v = v + 0.5 o v.astype(float)")

---
## 7️⃣ Operadores lógicos: `and`, `or`, `not`

| Operador | Resultado |
|:-:|---|
| `A and B` | verdadero si **ambos** lo son |
| `A or B` | verdadero si **al menos uno** lo es |
| `not A` | invierte el valor |

**Tabla de verdad:**

In [ ]:
print(f"{'A':^7}{'B':^7}{'A and B':^10}{'A or B':^9}{'not A':^7}")
for A in (True, False):
    for B in (True, False):
        print(f"{A!s:^7}{B!s:^7}{(A and B)!s:^10}{(A or B)!s:^9}{(not A)!s:^7}")

### 7.1 Cortocircuito (*short-circuit*) y valores "truthy/falsy"
- `A and B`: si `A` es falso, **no evalúa** `B`.
- `A or B`: si `A` es verdadero, **no evalúa** `B`.
- Devuelven **uno de los operandos**, no necesariamente un `bool`.

In [ ]:
def evalua(nombre, valor):
    print(f"   (evaluando {nombre})")
    return valor

print("False and ...:", evalua("A", False) and evalua("B", True))   # B nunca se evalúa
print("True  or  ...:", evalua("A", True) or evalua("B", False))    # B nunca se evalúa

# Uso práctico: evitar división por cero
n, total = 0, 100
promedio = n > 0 and total / n
print("\npromedio:", promedio)

# Valor por defecto con `or`
config = {"lr": None}
lr = config["lr"] or 0.01
print("lr:", lr)

# ⚠️ Pero cuidado: un 0 legítimo también es "falsy"
config = {"dropout": 0.0}
print("dropout mal  :", config["dropout"] or 0.5)                 # 0.5 ← ¡error de lógica!
print("dropout bien :", config.get("dropout") if config.get("dropout") is not None else 0.5)

### 7.2 Regla de parada temprana (*early stopping*)
Combinamos comparación + lógicos para decidir cuándo detener el entrenamiento.

In [ ]:
def debe_detenerse(epoch, val_loss, mejor_loss, sin_mejora, paciencia=3, min_epochs=5):
    """Detiene si superamos la paciencia Y ya corrimos las épocas mínimas,
    o si la pérdida se volvió NaN/infinita (entrenamiento divergió)."""
    divergio = math.isnan(val_loss) or math.isinf(val_loss)
    sin_progreso = sin_mejora >= paciencia and epoch >= min_epochs
    return divergio or sin_progreso

print(debe_detenerse(epoch=8, val_loss=0.40, mejor_loss=0.35, sin_mejora=3))   # True
print(debe_detenerse(epoch=3, val_loss=0.40, mejor_loss=0.35, sin_mejora=3))   # False
print(debe_detenerse(epoch=2, val_loss=float("nan"), mejor_loss=0.35, sin_mejora=0))  # True

### 7.3 ⚠️ En NumPy y pandas **no** se usa `and`/`or`/`not`
Para combinar máscaras elemento a elemento se usan `&`, `|`, `~` **con paréntesis** (por la precedencia), o `np.logical_and/or/not`.

| Python puro | NumPy / pandas |
|:-:|:-:|
| `and` | `&` |
| `or` | `\|` |
| `not` | `~` |

In [ ]:
edad   = np.array([15, 22, 37, 45, 17, 60])
activo = np.array([True, True, False, True, False, True])

try:
    mask = (edad >= 18) and activo
except ValueError as e:
    print("ValueError:", str(e)[:75], "...\n")

mayores_activos = (edad >= 18) & activo
print("adultos Y activos :", mayores_activos)
print("jóvenes O inactivos:", (edad < 18) | ~activo)
print("equivalente        :", np.logical_and(edad >= 18, activo))
print("\nany / all →", (edad > 50).any(), (edad > 10).all())

# En pandas funciona igual
df = pd.DataFrame({"area": [45, 80, 120, 60, 200],
                   "habitaciones": [1, 2, 3, 2, 5],
                   "ciudad": ["Cali", "Medellín", "Bogotá", "Medellín", "Cali"]})
filtro = (df["area"] > 50) & (df["habitaciones"] <= 3) & ~(df["ciudad"] == "Cali")
display(df[filtro])

---
## 8️⃣ Operadores de pertenencia e identidad

### 8.1 Pertenencia: `in` y `not in`
Verifican si un elemento **está contenido** en una colección (en `dict` se buscan las **claves**; en `str`, subcadenas).

In [ ]:
activaciones_validas = ("relu", "tanh", "sigmoid", "gelu")
hiper = {"lr": 0.001, "batch": 32}

print("relu" in activaciones_validas)          # True
print("softmax" not in activaciones_validas)   # True
print("lr" in hiper, "momentum" in hiper)      # True False  (busca claves)
print(0.001 in hiper.values())                 # True        (busca valores)
print("model" in "modelo de lenguaje")         # True        (subcadena)

# Validar entradas del usuario
def crear_capa(activacion):
    if activacion not in activaciones_validas:
        raise ValueError(f"Activación '{activacion}' no soportada. Opciones: {activaciones_validas}")
    return f"Capa(activación={activacion})"

print(crear_capa("gelu"))
try:
    crear_capa("softmax")
except ValueError as e:
    print("Error →", e)

#### Aplicación en NLP: vocabulario y palabras fuera de vocabulario (*OOV*)

In [ ]:
vocab = {"<pad>": 0, "<unk>": 1, "el": 2, "modelo": 3, "aprende": 4, "datos": 5}
frase = "el modelo aprende de los datos".split()

ids = [vocab[t] if t in vocab else vocab["<unk>"] for t in frase]
print(frase)
print(ids)                                   # 'de' y 'los' → 1 (<unk>)

desconocidas = [t for t in frase if t not in vocab]
print("OOV:", desconocidas)

#### Rendimiento: `in` en `list` vs `set`/`dict`
- En una **lista** la búsqueda es **O(n)** (recorre elemento a elemento).
- En un **set/dict** es **O(1)** promedio (tabla hash).

Con vocabularios de cientos de miles de palabras, la diferencia es enorme.

In [ ]:
N = 200_000
lista, conjunto = list(range(N)), set(range(N))
objetivo = N - 1                      # peor caso para la lista

t0 = time.perf_counter()
for _ in range(100): objetivo in lista
t_lista = time.perf_counter() - t0

t0 = time.perf_counter()
for _ in range(100): objetivo in conjunto
t_set = time.perf_counter() - t0

print(f"list: {t_lista*1000:8.2f} ms")
print(f"set : {t_set*1000:8.4f} ms")
print(f"→ el set fue ~{t_lista / max(t_set, 1e-9):,.0f}× más rápido")

# Pertenencia vectorizada
etiquetas = np.array(["gato", "perro", "ave", "gato", "pez"])
print("\nnp.isin :", np.isin(etiquetas, ["gato", "ave"]))
print("pandas  :", pd.Series(etiquetas).isin(["gato", "ave"]).tolist())

### 8.2 Identidad: `is` y `is not`
`==` compara **valores**; `is` compara si son **el mismo objeto en memoria** (`id`).

> ✅ Usa `is` / `is not` casi exclusivamente con `None`, `True` y `False`.

In [ ]:
a = [1, 2, 3]
b = [1, 2, 3]
c = a

print("a == b :", a == b)       # True  → mismo contenido
print("a is b :", a is b)       # False → objetos distintos
print("a is c :", a is c)       # True  → es el mismo objeto
print("id(a), id(b), id(c):", id(a), id(b), id(c))

# Patrón correcto para comprobar None
modelo = None
if modelo is None:
    print("\nAún no se ha cargado el modelo")

# Un caso donde == falla pero `is` no: arreglos NumPy
arr = np.array([1, 2])
print("arr is None →", arr is None)      # bool simple. (arr == None daría un arreglo)

---
## 9️⃣ Operadores bit a bit y precedencia

### 9.1 Operadores bit a bit
Actúan sobre la representación **binaria** de los enteros.

| Operador | Nombre | Ejemplo (`a=5`→`101`, `b=3`→`011`) |
|:-:|---|---|
| `&` | AND | `5 & 3 = 1` (`001`) |
| `\|` | OR | `5 \| 3 = 7` (`111`) |
| `^` | XOR | `5 ^ 3 = 6` (`110`) |
| `~` | NOT | `~5 = -6` |
| `<<` | desplazar izq. | `5 << 1 = 10` (×2) |
| `>>` | desplazar der. | `5 >> 1 = 2` (÷2 entero) |

Son poco frecuentes en código de ML "de alto nivel", pero aparecen en **máscaras de bits**, *hashing de características*, optimización de modelos (cuantización) y bajo nivel en GPU.

In [ ]:
a, b = 5, 3
print(f"a={a:03b}  b={b:03b}")
print(f"a & b = {a & b:03b} ({a & b})")
print(f"a | b = {a | b:03b} ({a | b})")
print(f"a ^ b = {a ^ b:03b} ({a ^ b})")
print(f"~a    = {~a}")
print(f"a << 1 = {a << 1} | 16 >> 2 = {16 >> 2}")

# Banderas de configuración empaquetadas en un entero
USA_BIAS    = 1 << 0     # 001
USA_DROPOUT = 1 << 1     # 010
USA_BATCHN  = 1 << 2     # 100

cfg = USA_BIAS | USA_BATCHN                   # activar banderas
print(f"\ncfg = {cfg:03b}")
print("¿dropout activo?  ", bool(cfg & USA_DROPOUT))
print("¿batchnorm activo?", bool(cfg & USA_BATCHN))
cfg ^= USA_DROPOUT                            # alternar bandera
print(f"cfg tras alternar dropout = {cfg:03b}")

#### Aplicación en Deep Learning: **Dropout** con máscara booleana
Durante el entrenamiento se "apagan" neuronas al azar con probabilidad $p$ y se reescalan las restantes por $\dfrac{1}{1-p}$ (*inverted dropout*). Solo usa comparación, `*` y `/`.

In [ ]:
p = 0.4
activaciones = rng.normal(size=10).round(2)
mascara = rng.random(10) >= p                    # True = la neurona sobrevive
salida = activaciones * mascara / (1 - p)

print("activaciones:", activaciones)
print("máscara     :", mascara.astype(int))
print("salida      :", salida.round(2))

### 9.2 Precedencia de operadores (de **mayor** a **menor**)

| Nivel | Operadores |
|:-:|---|
| 1 | `( )` paréntesis |
| 2 | `**` |
| 3 | `+x`, `-x`, `~x` (unarios) |
| 4 | `*`, `/`, `//`, `%`, `@` |
| 5 | `+`, `-` |
| 6 | `<<`, `>>` |
| 7 | `&` |
| 8 | `^` |
| 9 | `\|` |
| 10 | `==`, `!=`, `<`, `>`, `<=`, `>=`, `is`, `is not`, `in`, `not in` |
| 11 | `not` |
| 12 | `and` |
| 13 | `or` |
| 14 | `:=` y asignaciones (`=`, `+=`, …) |

> 📌 **Regla de oro:** ante la duda, **usa paréntesis**. Mejora la legibilidad y evita errores.

In [ ]:
print("2 + 3 * 4 ** 2   =", 2 + 3 * 4 ** 2)          # 2 + 3*16 = 50
print("-2 ** 2           =", -2 ** 2)                  # -(2**2) = -4  ← ¡no es 4!
print("(-2) ** 2         =", (-2) ** 2)
print("not 1 == 2        =", not 1 == 2)               # not (1 == 2) = True
print("True or False and False =", True or False and False)   # and antes que or → True

# Trampa con máscaras NumPy: & tiene MAYOR precedencia que >
x = np.array([1, 5, 10])
try:
    print(x > 2 & x < 8)             # se interpreta como x > (2 & x) < 8
except ValueError:
    print("\n¡Sin paréntesis falla! →  (x > 2) & (x < 8) =", (x > 2) & (x < 8))

---
## 🔟 🧪 Mini-proyecto integrador: regresión lineal desde cero

**Problema:** predecir el **precio** de una vivienda (en millones de COP) a partir de su área, número de habitaciones y antigüedad. Entrenaremos el modelo $\hat{y} = \mathbf{X}\mathbf{w}$ con **descenso del gradiente**, usando solo lo visto en la clase:

| Paso | Operadores / tipos que practicamos |
|---|---|
| 1. Generar datos sucios | tipos NumPy, `float`, `NaN`, `int` |
| 2. Limpieza | comparación, lógicos (`&`, `~`), máscaras |
| 3. Normalización | aritméticos (`-`, `/`), z-score |
| 4. Partición train/val | `//`, `*`, slicing |
| 5. Entrenamiento | `@`, `**`, `-=`, `%`, early stopping (`and`/`or`) |
| 6. Métricas | comparación, `&`, conteos con `bool`, `in` |

#### Paso 1 · Generar un dataset sintético con "suciedad"

In [ ]:
n = 500
area   = rng.normal(80, 25, n).clip(20, None)     # m²
habit  = rng.integers(1, 6, n)                    # 1–5
antig  = rng.integers(0, 50, n)                   # años
ruido  = rng.normal(0, 10, n)

# Relación "verdadera" (desconocida para el modelo)
precio = 1.5 * area + 12 * habit - 0.8 * antig + 20 + ruido

X = np.column_stack([area, habit, antig]).astype(np.float64)
y = precio.copy()

# Ensuciamos los datos, como en la vida real
X[rng.choice(n, 10, replace=False), 0] = np.nan       # 10 áreas faltantes
y[rng.choice(n, 5, replace=False)]     = -999         # 5 precios imposibles (código de error)
fuera = rng.choice(n, 3, replace=False)
y[fuera] = y[fuera] * 50                              # 3 valores atípicos extremos

print("Forma X:", X.shape, "| dtype:", X.dtype)
print("NaN en X      :", np.isnan(X).sum())
print("Precios <= 0  :", (y <= 0).sum())
print("Precios > 1000:", (y > 1000).sum())

#### Paso 2 · Limpieza con comparaciones y operadores lógicos

In [ ]:
sin_nan      = ~np.isnan(X).any(axis=1)         # filas sin ningún NaN
precio_valido = (y > 0) & (y < 1000)            # rango plausible
mascara_ok   = sin_nan & precio_valido

Xc, yc = X[mascara_ok], y[mascara_ok]
print(f"Filas originales: {len(y)} | conservadas: {len(yc)} | descartadas: {(~mascara_ok).sum()}")
print(f"Precio medio: {yc.mean():.1f} | mín: {yc.min():.1f} | máx: {yc.max():.1f}")

#### Paso 3 · Normalización (z-score) con operadores aritméticos
> Guardamos $\mu$ y $\sigma$ porque **hay que reutilizarlos** con datos nuevos en producción.

In [ ]:
mu_x, sd_x = Xc.mean(axis=0), Xc.std(axis=0)
mu_y, sd_y = yc.mean(), yc.std()

Xn = (Xc - mu_x) / sd_x
yn = (yc - mu_y) / sd_y

print("media de Xn ≈", Xn.mean(axis=0).round(6))
print("desv. de Xn ≈", Xn.std(axis=0).round(6))

#### Paso 4 · Partición entrenamiento / validación (80 / 20)

In [ ]:
m = len(yn)
idx = rng.permutation(m)              # barajar índices
corte = int(0.8 * m)                  # int() trunca

tr, va = idx[:corte], idx[corte:]

# Añadimos columna de unos para el sesgo (bias): y = w0 + w1*x1 + ...
Xb = np.c_[np.ones(m), Xn]
Xtr, ytr = Xb[tr], yn[tr]
Xva, yva = Xb[va], yn[va]
print(f"Entrenamiento: {len(tr)} | Validación: {len(va)} | Forma Xtr: {Xtr.shape}")

#### Paso 5 · Entrenamiento con descenso del gradiente + *early stopping*
El gradiente del MSE respecto a $\mathbf{w}$ es $\;\nabla = \dfrac{2}{n}\,\mathbf{X}^\top(\mathbf{X}\mathbf{w}-\mathbf{y})$

In [ ]:
config = {"lr": 0.1, "epochs": 1000, "paciencia": 20, "tol": 1e-6}

# Pertenencia entre conjuntos: ¿están todas las claves requeridas?
requeridas = {"lr", "epochs", "paciencia", "tol"}
assert requeridas <= set(config), f"Faltan claves: {requeridas - set(config)}"

w = np.zeros(Xtr.shape[1])
mejor_loss, sin_mejora, mejor_w = float("inf"), 0, w.copy()
hist_tr, hist_va = [], []

for epoch in range(1, config["epochs"] + 1):
    # --- paso hacia adelante y gradiente ---
    error = Xtr @ w - ytr
    grad  = (2 / len(ytr)) * (Xtr.T @ error)
    w    -= config["lr"] * grad                       # actualización

    # --- pérdidas ---
    loss_tr = ((Xtr @ w - ytr) ** 2).mean()
    loss_va = ((Xva @ w - yva) ** 2).mean()
    hist_tr.append(loss_tr); hist_va.append(loss_va)

    # --- early stopping ---
    if loss_va < mejor_loss - config["tol"]:
        mejor_loss, mejor_w, sin_mejora = loss_va, w.copy(), 0
    else:
        sin_mejora += 1

    if epoch == 1 or epoch % 50 == 0:
        print(f"época {epoch:4d} | loss_train={loss_tr:.5f} | loss_val={loss_va:.5f}")

    if sin_mejora >= config["paciencia"] and epoch > 10:
        print(f"\n⏹  Early stopping en la época {epoch} (mejor val_loss = {mejor_loss:.5f})")
        break

w = mejor_w
print("\nPesos aprendidos (normalizados) [bias, área, habitaciones, antigüedad]:")
print(w.round(4))

In [ ]:
plt.figure(figsize=(7, 3.5))
plt.plot(hist_tr, label="entrenamiento")
plt.plot(hist_va, label="validación")
plt.yscale("log")
plt.xlabel("época"); plt.ylabel("MSE (escala log)")
plt.title("Curvas de aprendizaje"); plt.legend(); plt.grid(alpha=.3)
plt.tight_layout(); plt.show()

#### Paso 6 · Evaluación: regresión y clasificación derivada
Convertimos las predicciones a la escala original (¡deshacer la normalización!) y calculamos métricas.

In [ ]:
pred_va_real = (Xva @ w) * sd_y + mu_y          # desnormalizar
y_va_real    = yva * sd_y + mu_y

rmse = (((pred_va_real - y_va_real) ** 2).mean()) ** 0.5
ss_res = ((y_va_real - pred_va_real) ** 2).sum()
ss_tot = ((y_va_real - y_va_real.mean()) ** 2).sum()
r2 = 1 - ss_res / ss_tot
print(f"RMSE = {rmse:.2f} millones | R² = {r2:.4f}")

# Pesos en unidades originales (comparar con la relación verdadera: 1.5, 12, -0.8)
coef_real = w[1:] * sd_y / sd_x
print("\nCoeficientes recuperados [área, habit., antig.]:", coef_real.round(2), "(reales: 1.5, 12, -0.8)")

# Clasificación derivada: ¿vivienda "cara" (> mediana)?
umbral = np.median(y_va_real)
real_caro = y_va_real > umbral
pred_caro = pred_va_real > umbral

tp = (pred_caro & real_caro).sum()
fp = (pred_caro & ~real_caro).sum()
fn = (~pred_caro & real_caro).sum()
tn = (~pred_caro & ~real_caro).sum()

precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
accuracy  = (tp + tn) / len(real_caro)
print(f"\nMatriz de confusión → TP={tp} FP={fp} FN={fn} TN={tn}")
print(f"accuracy={accuracy:.2%} | precision={precision:.2%} | recall={recall:.2%}")

### 🎓 ¿Qué operadores usamos en el proyecto?
Revisa el código y localiza cada familia: **aritméticos** (`-`, `/`, `*`, `**`, `@`), **comparación** (`>`, `<`, `>=`), **lógicos** (`&`, `~`, `and`), **asignación** (`-=`, `+=`, `=`), **pertenencia** (`<=` entre sets) y **módulo** (`%`) para el *logging*. ¡Un modelo completo sin librerías de ML!

---
## 1️⃣1️⃣ Errores comunes (checklist de supervivencia)

| ❌ Error | ✅ Corrección |
|---|---|
| `0.1 + 0.2 == 0.3` | `math.isclose(...)` / `np.allclose(...)` |
| `x == np.nan` | `np.isnan(x)` / `pd.isna(x)` |
| `if arr1 == arr2:` | `np.array_equal(arr1, arr2)`, `.all()`, `.any()` |
| `(a > 1) and (a < 5)` con arreglos | `(a > 1) & (a < 5)` |
| `a > 1 & a < 5` (sin paréntesis) | `(a > 1) & (a < 5)` |
| `b = a` creyendo que copia | `b = a.copy()` / `copy.deepcopy(a)` |
| `x is 5` / `x == None` | `x == 5` / `x is None` |
| `imagen_uint8 + 50` | `imagen.astype(np.float32) + 50` |
| `valor or default` con `0` legítimo | `default if valor is None else valor` |
| `int("3.7")` | `int(float("3.7"))` |
| `x in lista_grande` en un bucle | convertir a `set` antes |

## 1️⃣2️⃣ Autoevaluación rápida
**Predice el resultado** de cada expresión antes de abrir la respuesta.

1. `10 // 3 + 10 % 3`
2. `not (5 > 3 and 2 > 4)`
3. `[1, 2] * 2 == [1, 2, 1, 2]`
4. `type(True + 1).__name__`
5. `"a" in {"a": 1, "b": 2}`
6. `0.1 * 3 == 0.3`
7. `-3 ** 2`
8. `(7 & 3) | 8`

<details>
<summary><b>📖 Ver respuestas</b></summary>

1. `4` (`10//3=3`, `10%3=1`)
2. `True` (`5>3 and 2>4` es `False`, y `not False` es `True`)
3. `True`
4. `'int'` (`True + 1 = 2`)
5. `True` (busca en las claves)
6. `False` (error de precisión de flotantes)
7. `-9` (la potencia se evalúa antes que el menos unario)
8. `11` (`7&3=3`; `3|8=11`)
</details>

In [ ]:
# Verifica tus respuestas ejecutando esta celda
print(1, 10 // 3 + 10 % 3)
print(2, not (5 > 3 and 2 > 4))
print(3, [1, 2] * 2 == [1, 2, 1, 2])
print(4, type(True + 1).__name__)
print(5, "a" in {"a": 1, "b": 2})
print(6, 0.1 * 3 == 0.3)
print(7, -3 ** 2)
print(8, (7 & 3) | 8)

## 1️⃣3️⃣ 📝 Ejercicios prácticos
Resuélvelos en las celdas vacías. Las soluciones están al final del cuaderno (**intenta primero por tu cuenta**).

---
**Ejercicio 1 · Conversión y estandarización** 🌡️
Dada la lista de temperaturas de un sensor (texto, con datos corruptos):
`["21.5", "23.0", "ERROR", "19.8", "", "25.1"]`
(a) conviértela a `float` usando `nan` para los inválidos; (b) calcula la media ignorando los `nan` con `np.nanmean`; (c) conviértela a Fahrenheit con `F = C * 9/5 + 32`.

In [ ]:
crudo = ["21.5", "23.0", "ERROR", "19.8", "", "25.1"]
# Tu código aquí

**Ejercicio 2 · Batches** 📦
Un dataset tiene **10 837** imágenes y usas `batch_size = 64`.
Calcula: (a) batches completos, (b) imágenes que sobran, (c) iteraciones por época si **no** descartas el último batch, (d) iteraciones totales en 12 épocas, (e) en qué épocas (de 1 a 12) se guardaría un checkpoint si guardas cada 4 épocas.

In [ ]:
n_imagenes, batch_size, epocas = 10_837, 64, 12
# Tu código aquí

**Ejercicio 3 · Máscaras y accuracy** 🎯
Dados `probs` (probabilidades del modelo) y `y_true`, clasifica como positivo si `prob >= 0.6`, calcula la *accuracy* y cuántos falsos negativos hay (real = 1, predicho = 0).

In [ ]:
probs  = np.array([0.95, 0.40, 0.65, 0.55, 0.80, 0.10, 0.60, 0.35])
y_true = np.array([1,    0,    1,    1,    1,    0,    0,    1   ])
# Tu código aquí

**Ejercicio 4 · Pertenencia** 🔍
Tienes las columnas de un dataset y las que tu modelo requiere. Usando `in`/`not in` (o conjuntos) determina: (a) las columnas requeridas que **faltan**, (b) las columnas **sobrantes**, (c) si el dataset puede usarse (todas las requeridas presentes).

In [ ]:
columnas_dataset   = ["id", "edad", "ingreso", "ciudad", "fecha"]
columnas_requeridas = ["edad", "ingreso", "genero", "ciudad"]
# Tu código aquí

**Ejercicio 5 · Regla de decisión con lógicos** ⚙️
Escribe `puede_desplegarse(accuracy, recall, val_loss, train_loss)` que devuelva `True` solo si:
`accuracy >= 0.90` **y** `recall >= 0.85` **y** que no haya sobreajuste severo, es decir `(val_loss - train_loss) < 0.15`; y además ningún valor debe ser `NaN`.

In [ ]:
def puede_desplegarse(accuracy, recall, val_loss, train_loss):
    # Tu código aquí
    pass

# Pruebas (cuando termines, deberían imprimir True, False, False, False)
print(puede_desplegarse(0.93, 0.88, 0.30, 0.22))
print(puede_desplegarse(0.93, 0.80, 0.30, 0.22))
print(puede_desplegarse(0.93, 0.88, 0.60, 0.22))
print(puede_desplegarse(float("nan"), 0.88, 0.30, 0.22))

**Ejercicio 6 · 🧠 Un perceptrón** (reto)
En 1958 Rosenblatt propuso el perceptrón: $\hat{y} = 1$ si $\mathbf{x}\cdot\mathbf{w} + b > 0$, y $0$ en caso contrario.
Encuentra pesos y sesgo que implementen las compuertas **AND** y **OR**, y comprueba con la tabla de verdad usando `@`, `+` y `>`.
*(Pista: para AND, $w=[1,1]$ y $b=-1.5$.)*

In [ ]:
X_logica = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
# Tu código aquí

---
## ✅ Soluciones (solo para consulta después de intentarlo)

In [ ]:
# ----- Solución Ejercicio 1 -----
crudo = ["21.5", "23.0", "ERROR", "19.8", "", "25.1"]
celsius = []
for dato in crudo:
    try:
        celsius.append(float(dato))
    except ValueError:
        celsius.append(float("nan"))
celsius = np.array(celsius)
print("celsius   :", celsius)
print("media     :", np.nanmean(celsius).round(3))
print("fahrenheit:", (celsius * 9 / 5 + 32).round(2))

In [ ]:
# ----- Solución Ejercicio 2 -----
n_imagenes, batch_size, epocas = 10_837, 64, 12
completos = n_imagenes // batch_size
sobran    = n_imagenes % batch_size
iter_epoca = math.ceil(n_imagenes / batch_size)
iter_total = iter_epoca * epocas
checkpoints = [e for e in range(1, epocas + 1) if e % 4 == 0]
print(completos, sobran, iter_epoca, iter_total, checkpoints)
assert (completos, sobran, iter_epoca, iter_total) == (169, 21, 170, 2040)
assert checkpoints == [4, 8, 12]

In [ ]:
# ----- Solución Ejercicio 3 -----
probs  = np.array([0.95, 0.40, 0.65, 0.55, 0.80, 0.10, 0.60, 0.35])
y_true = np.array([1,    0,    1,    1,    1,    0,    0,    1   ])
y_hat = (probs >= 0.6).astype(int)
accuracy = (y_hat == y_true).mean()
falsos_negativos = ((y_true == 1) & (y_hat == 0)).sum()
print("predicción:", y_hat)
print(f"accuracy = {accuracy:.2%} | falsos negativos = {falsos_negativos}")
assert falsos_negativos == 2

In [ ]:
# ----- Solución Ejercicio 4 -----
columnas_dataset    = ["id", "edad", "ingreso", "ciudad", "fecha"]
columnas_requeridas = ["edad", "ingreso", "genero", "ciudad"]
faltan   = [c for c in columnas_requeridas if c not in columnas_dataset]
sobran   = [c for c in columnas_dataset if c not in columnas_requeridas]
utilizable = len(faltan) == 0
print("faltan:", faltan, "| sobran:", sobran, "| ¿utilizable?", utilizable)
# Versión con conjuntos:
print(set(columnas_requeridas) - set(columnas_dataset), set(columnas_dataset) - set(columnas_requeridas))
assert faltan == ["genero"] and not utilizable

In [ ]:
# ----- Solución Ejercicio 5 -----
def puede_desplegarse(accuracy, recall, val_loss, train_loss):
    hay_nan = any(math.isnan(v) for v in (accuracy, recall, val_loss, train_loss))
    return (not hay_nan) and accuracy >= 0.90 and recall >= 0.85 and (val_loss - train_loss) < 0.15

res = [puede_desplegarse(0.93, 0.88, 0.30, 0.22),
       puede_desplegarse(0.93, 0.80, 0.30, 0.22),
       puede_desplegarse(0.93, 0.88, 0.60, 0.22),
       puede_desplegarse(float("nan"), 0.88, 0.30, 0.22)]
print(res)
assert res == [True, False, False, False]

In [ ]:
# ----- Solución Ejercicio 6 -----
X_logica = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
compuertas = {
    "AND": (np.array([1.0, 1.0]), -1.5),
    "OR" : (np.array([1.0, 1.0]), -0.5),
    "NAND": (np.array([-1.0, -1.0]), 1.5),
}
for nombre, (w_, b_) in compuertas.items():
    salida = (X_logica @ w_ + b_ > 0).astype(int)
    print(f"{nombre:5s}", salida)

# ¡Reto extra! XOR NO es linealmente separable: ningún (w, b) funciona con una sola neurona.
# Por eso nacieron las redes con capas ocultas (perceptrón multicapa).

---
## 🧾 Resumen de la clase (hoja de referencia rápida)

```python
# TIPOS         int  float  complex  bool  str  None  |  list  tuple  set  dict  range
# NumPy dtypes  float64 float32 float16 int64 uint8 bool_   → .astype(...)

# ARITMÉTICOS   +  -  *  /  //  %  **  @
# COMPARACIÓN   ==  !=  >  <  >=  <=            (encadenable: 0 <= p <= 1)
# ASIGNACIÓN    =  +=  -=  *=  /=  //=  %=  **=  :=
# LÓGICOS       and  or  not        (arreglos: &  |  ~  con paréntesis)
# PERTENENCIA   in  not in          (set/dict → O(1))
# IDENTIDAD     is  is not          (úsalo con None)
# BIT A BIT     &  |  ^  ~  <<  >>
```

### 🔑 Ideas clave
1. **El tipo de dato importa**: `float32` vs `float64`, `uint8` vs `float` y `bool` como `int` afectan memoria, precisión y resultados.
2. **Los operadores cambian de significado según el tipo** (`list` vs `ndarray`).
3. **Comparar flotantes y `NaN` requiere cuidado**; usa `isclose`, `isnan`.
4. **Las máscaras booleanas** (comparación + `&`, `|`, `~`) son la base del filtrado y la limpieza de datos.
5. **Asignar no es copiar** cuando el objeto es mutable.
6. Con apenas operadores se construye todo el ciclo de entrenamiento: normalizar → predecir (`@`) → perder (`**`, `mean`) → actualizar (`-=`).

---
## 📚 Para profundizar
- Documentación oficial de Python — *Expressions* y *Built-in Types*: https://docs.python.org/3/reference/expressions.html
- NumPy — *Data types*: https://numpy.org/doc/stable/user/basics.types.html
- pandas — *User Guide*: https://pandas.pydata.org/docs/user_guide/index.html
- Aurélien Géron, *Hands-On Machine Learning with Scikit-Learn, Keras & TensorFlow*.
- Jake VanderPlas, *Python Data Science Handbook* (gratuito en línea).

## 🏠 Tarea para la próxima clase
1. Descarga un dataset real (p. ej. *Titanic* o *Iris* desde seaborn/scikit-learn) y **reporta el `dtype` de cada columna**, la cantidad de `NaN` y 3 filtros con máscaras compuestas (`&`, `|`, `~`).
2. Normaliza (z-score) dos columnas numéricas **sin usar `sklearn`**, solo operadores.
3. Implementa **regresión logística** (sigmoide + entropía cruzada) con descenso del gradiente reutilizando el código del mini-proyecto.

> **Próxima clase (3):** Estructuras de control (condicionales y bucles) y funciones aplicadas a pipelines de datos. 🚀